In [1]:
import os
import argparse
import math
from copy import deepcopy

import numpy as np

import torch
import torch.backends.cudnn as cudnn
import torchvision.models as models
from torchvision.models import resnet50

import clip
import json
from tqdm.notebook import tqdm
from pkg_resources import packaging
from imagenetv2_pytorch import ImageNetV2Dataset

import sys
sys.path.insert(0,'/home/jovyan/CLIP')

from lib.utils.utils import prYellow
from lib.env import ImportantBitsFloatFirstNEnv, ImportantBitsQuanFirstNEnv
from lib.ddpg import DDPG

#pip install git+https://github.com/openai/CLIP.git
#pip install cupy-cuda11x
#pip install git+https://github.com/modestyachts/ImageNetV2_pytorch

/tmp/ipykernel_433/3984193861.py:16: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import packaging


In [2]:
model_names = ['CLIP']
print('support models: ', model_names)

support models:  ['CLIP']


In [3]:
def train(num_episode, agent, env, output, args, debug=False):
    # best record
    best_reward = -math.inf
    best_policy = []
    acc = 0.0
    ratio = 0.0

    agent.is_training = True
    step = episode = episode_steps = 0
    episode_reward = 0.
    observation = None
    T = []  # trajectory
    while episode < num_episode:  # counting based on episode
        # reset if it is the start of episode
        if observation is None:
            observation = deepcopy(env.reset())
            agent.reset(observation)
            if episode > args.warmup:
                agent.step(episode in [750, 1500])
                # agent.step(episode in [])

        # agent pick action ...
        if episode <= args.warmup:
            action = agent.random_action()
            #action = agent.baseline_action(env.mask_hint)
            # action = agent.baseline_action(get_mask([0, 1]))
        else:
            action = agent.select_action(observation)

        # env response with next_observation, reward, terminate_info
        observation2, reward, done, info = env.step(action)
        observation2 = deepcopy(observation2)
        T.append([reward, deepcopy(observation), deepcopy(observation2), action, done])

        # [optional] save intermideate model
        if episode % int(num_episode / 10) == 0:
            agent.save_model(output)

        # update
        step += 1
        episode_steps += 1
        episode_reward += reward
        observation = deepcopy(observation2)

        if done:  # end of episode
            debug = True
            if debug:
                print('#{}: episode_reward:{:.4f} acc: {:.4f}, weight: {:.4f} MB'.format(episode, episode_reward,
                                                                                         info['accuracy'],
                                                                                         info['w_size'] * 1. / 8e6))
            text_writer.write(
                '#{}: episode_reward:{:.4f} acc: {:.4f}, weight: {:.4f} MB\n'.format(episode, episode_reward,
                                                                                     info['accuracy'],
                                                                                     info['w_size'] * 1. / 8e6))
            for i in range(len(T) - 1):
                T[i+1][1][-env.n_actions:] = env.strategy[i]
            final_reward = T[-1][0]
            # agent observe and update policy
            for i, (r_t, s_t, s_t1, a_t, done) in enumerate(T):
                agent.observe(final_reward, s_t, a_t, done)
                if episode > args.warmup:
                    for i in range(args.n_update):
                        agent.update_policy()

            agent.memory.append(
                observation,
                agent.select_action(observation),
                0., False
            )

            # test
            if episode % 50 == 0:
                test(5, agent, env, debug)

            # reset
            observation = None
            episode_steps = 0
            episode_reward = 0.
            episode += 1
            T = []

            if final_reward > best_reward:
                best_reward = final_reward
                best_policy = env.strategy
                best_episode_num = episode
                acc = info['accuracy']
                ratio = info['w_size'] * 1. / 8e6

            text_writer.write('best_reward: {}\n'.format(best_reward))
            text_writer.write('best_policy: {}\n'.format(best_policy))
    text_writer.write('best_accuracy: {}\n'.format(acc))
    text_writer.write('best_ratio: {}\n'.format(ratio))
    text_writer.write('best_episode_num: {}\n'.format(best_episode_num))
    text_writer.close()
    return best_policy, best_reward, acc, ratio

def test(num_episode, agent, env, debug=False):

    agent.is_training = False
    step = episode = episode_steps = 0
    episode_reward = 0.
    observation = None
    while episode < num_episode:  # counting based on episode
        # reset if it is the start of episode
        if observation is None:
            observation = deepcopy(env.reset())

        action = agent.select_action(observation)

        observation2, reward, done, info = env.step(action)
        observation2 = deepcopy(observation2)

        step += 1
        episode_steps += 1
        episode_reward += reward
        observation = deepcopy(observation2)

        if done:  # end of episode
            prYellow('#Test{}: episode_reward:{:.4f} acc: {:.4f}, weight: {:.4f} MB'.format(episode, episode_reward,
                                                                                            info['accuracy'],
                                                                                            info['w_ratio'] * 1. / 8e6))

            # reset
            observation = None
            episode_steps = 0
            episode_reward = 0.
            episode += 1

    agent.is_training = True

In [4]:

if __name__ == "__main__":
    parser = argparse.ArgumentParser(description='PyTorch Reinforcement Learning')

    parser.add_argument('--suffix', default=None, type=str, help='suffix to help you remember what experiment you ran')
    # env
    parser.add_argument('--dataset', default='ImageNetV2', type=str, help='dataset to use)')
    parser.add_argument('--dataset_root', default='data/imagenet', type=str, help='path to dataset)')
    parser.add_argument('--preserve_ratio', default=0.5, type=float, help='preserve ratio of the model size')
    parser.add_argument('--min_bit', default=0, type=int, help='minimum bit to use')
    parser.add_argument('--max_bit', default=8, type=int, help='minimum bit to use')
    parser.add_argument('--n_actions', default=10, type=int, help='size of actions')
    parser.add_argument('--bit', default=8, type=int, help='bitwidth of model to use')
    parser.add_argument('--float_bit', default=32, type=int, help='the bit of full precision float')
    parser.add_argument('--is_pruned', dest='is_pruned', action='store_true')
    parser.add_argument('--representation', default='fixed', type=str, help='binary weight representation', choices=['float', 'fixed'])
    parser.add_argument('--code', default='bch', type=str, help='ECC code to use', choices=['ideal', 'bch'])
    # ddpg
    parser.add_argument('--hidden1', default=300, type=int, help='hidden num of first fully connect layer')
    parser.add_argument('--hidden2', default=300, type=int, help='hidden num of second fully connect layer')
    parser.add_argument('--lr_c', default=1e-3, type=float, help='learning rate for actor')
    parser.add_argument('--lr_a', default=1e-4, type=float, help='learning rate for actor')
    parser.add_argument('--warmup', default=30, type=int,
                        help='time without training but only filling the replay memory')
    parser.add_argument('--discount', default=1., type=float, help='')
    parser.add_argument('--bsize', default=128, type=int, help='minibatch size')
    parser.add_argument('--rmsize', default=256, type=int, help='memory size for each layer')
    parser.add_argument('--window_length', default=1, type=int, help='')
    parser.add_argument('--tau', default=0.01, type=float, help='moving average for target network')
    # noise (truncated normal distribution)
    parser.add_argument('--init_delta', default=0.5, type=float,
                        help='initial variance of truncated normal distribution')
    parser.add_argument('--delta_decay', default=0.992, type=float,
                        help='delta decay during exploration')
    parser.add_argument('--n_update', default=1, type=int, help='number of rl to update each time')
    # training
    parser.add_argument('--max_episode_length', default=1e9, type=int, help='')
    parser.add_argument('--output', default='check/checkpoint_noise/', type=str, help='')
    parser.add_argument('--debug', dest='debug', action='store_true')
    parser.add_argument('--init_w', default=0.003, type=float, help='')
    parser.add_argument('--train_episode', default=3500, type=int, help='train iters each timestep')
    parser.add_argument('--epsilon', default=50000, type=int, help='linear decay of exploration policy')
    parser.add_argument('--seed', default=7, type=int, help='')
    parser.add_argument('--n_worker', default=16, type=int, help='number of data loader worker')
    parser.add_argument('--data_bsize', default=128, type=int, help='number of data batch size')
    parser.add_argument('--finetune_lr', default=0.001, type=float, help='finetune gamma')
    parser.add_argument('--finetune_epoch', default=1, type=int, help='')
    parser.add_argument('--use_top5', default=False, type=bool, help='whether to use top5 acc in reward')
    parser.add_argument('--train_size', default=20000, type=int, help='number of train data size')
    parser.add_argument('--val_size', default=30000, type=int, help='number of val data size')
    parser.add_argument('--resume', default='default', type=str, help='Resuming model path for testing')
    # Architecture
    parser.add_argument('--arch', '-a', metavar='ARCH', default='CLIP', choices=model_names,
                        help='model architecture:' + ' | '.join(model_names) + ' (default: CLIP)')
    # device options
    parser.add_argument('--gpu_id', default='7', type=str,
                        help='id(s) for CUDA_VISIBLE_DEVICES')

    args = parser.parse_args('')
    base_folder_name = '{}_{}'.format(args.arch, args.dataset)
    if args.suffix is not None:
        base_folder_name = base_folder_name + '_' + args.suffix
    args.output = os.path.join(args.output, base_folder_name)
    if not os.path.exists(args.output):
        os.mkdir(args.output)
    # tfwriter = SummaryWriter(logdir=args.output)
    text_writer = open(os.path.join(args.output, 'log.txt'), 'w')
    print('==> Output path: {}...'.format(args.output))

    # Use CUDA
    # os.environ['CUDA_VISIBLE_DEVICES'] = args.gpu_id
    assert torch.cuda.is_available(), 'CUDA is needed for CNN'

    if args.seed > 0:
        np.random.seed(args.seed)
        torch.manual_seed(args.seed)
        torch.cuda.manual_seed_all(args.seed)

    # model = models.__dict__[args.arch](pretrained=True)
    if args.dataset == "ImageNetV2":
        model_name = "ViT-B/32"
        model, preprocess = clip.load("ViT-B/32",jit=False)
        state_dict = model.state_dict()
    else:
        model = None
        state_dict = None
    model.load_state_dict(state_dict)


    model = torch.nn.DataParallel(model).cuda()
    model.cuda()

    print('    Total params: %.2fM' % (sum(p.numel() for p in model.parameters())/1000000.0))
    # cudnn.benchmark = True

    device = "cuda" if torch.cuda.is_available() else "cpu"

    if args.code == 'ideal':
        code = None
    elif args.code == 'bch':
        if args.representation == 'fixed':
            code = (8191, 6787, 110) # Fixed point
        elif args.representation == 'float':
            code = (8191, 6722, 115) # Floating Point

    if args.representation == 'float':
        env = ImportantBitsFloatFirstNEnv(model, preprocess,args.dataset, args.dataset_root,
                                          compress_ratio=args.preserve_ratio, num_workers=args.n_worker,
                                          batch_size=args.data_bsize, args=args, bitwidth=args.bit, is_model_pruned=args.is_pruned, code=code)
    elif args.representation == 'fixed':
        env = ImportantBitsQuanFirstNEnv(model, preprocess, args.dataset, args.dataset_root,
                                         compress_ratio=args.preserve_ratio, num_workers=args.n_worker,
                                         batch_size=args.data_bsize, args=args, bitwidth=args.bit, is_model_pruned=args.is_pruned, code=code)

    nb_states = env.layer_embedding.shape[1]
    nb_actions = 1 # actions for weight and activation quantization
    args.rmsize = args.rmsize * len(env.layer_idx)  # for each layer
    print('** Actual replay buffer size: {}'.format(args.rmsize))
    agent = DDPG(nb_states, nb_actions, len(env.layer_idx), args)

    best_policy, best_reward, acc, ratio = train(args.train_episode, agent, env, args.output, args, debug=args.debug)
    print('best_reward: ', best_reward)
    print('best_policy: ', best_policy)
    print('accuracy: ', acc)
    print('ratio: ', ratio)

==> Output path: check/checkpoint_noise/CLIP_ImageNetV2...
    Total params: 151.28M
get dataset
ImageNetV2
10000


/opt/conda/lib/python3.11/site-packages/torch/utils/data/dataloader.py:561: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 8, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


=> Final layer index: [2, 3, 8, 9, 11, 13, 14, 17, 18, 20, 22, 23, 26, 27, 29, 31, 32, 35, 36, 38, 40, 41, 44, 45, 47, 49, 50, 53, 54, 56, 58, 59, 62, 63, 65, 67, 68, 71, 72, 74, 76, 77, 80, 81, 83, 85, 86, 89, 90, 92, 94, 95, 98, 99, 101, 103, 104, 107, 108, 110, 112, 113, 114, 119, 120, 122, 124, 125, 128, 129, 131, 133, 134, 137, 138, 140, 142, 143, 146, 147, 149, 151, 152, 155, 156, 158, 160, 161, 164, 165, 167, 169, 170, 173, 174, 176, 178, 179, 182, 183, 185, 187, 188, 191, 192, 194, 196, 197, 200, 201, 203, 205, 206, 209, 210, 212, 214, 215, 218, 219, 221, 223, 224, 225, 226]
=> Final bound list: [(0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 8), (0, 